<a href="https://colab.research.google.com/github/Sciform/sciform-hwz-ai-in-controlling/blob/main/lecture_2/2_9_ts_genai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenAI for Time Series: Zero-Shot Sales Forecasting with the Foundation Model Chronos

In notebook *2_8* we **trained** a global deep learning model (NHITS) on the daily revenue of 10 products of an online shop.
In this notebook we use the **same data and the same test period**, but we do not train anything: we use a pretrained
**time series foundation model** – the time-series counterpart of a Large Language Model.

### What is a time series foundation model?

| | Large Language Model (e.g. GPT, Claude) | Time series foundation model (e.g. Chronos) |
|---|---|---|
| pretrained on | huge amounts of text | huge amounts of time series (energy, traffic, retail, weather, synthetic data, ...) |
| input | sequence of tokens (words) | sequence of past values (the *context*) |
| output | the next tokens | the next values – as a probability distribution (quantiles) |
| usage | prompt → answer, no training needed | history → forecast, **zero-shot**, no training needed |

**Chronos** (Amazon, Apache 2.0 license) is one of the most used open-source models of this kind:

* **Chronos-Bolt** – a T5 encoder-decoder transformer; the history is cut into *patches* (short windows), which are the "tokens" of the model.
  It forecasts several quantiles at once and is fast enough for a CPU.
* **Chronos-2** – the newest generation (120 M parameters). In addition it can
  * forecast a *group* of related time series jointly (**cross-learning** – a zero-shot "global model"), and
  * use **covariates**, e.g. known future information such as holidays or promotions.

Other free, open-source models you can try with the same approach: **TimesFM** (Google), **Tiny Time Mixers / TTM** (IBM Granite),
**Moirai** (Salesforce – check the license of the weights), **Lag-Llama**, **MOMENT**. *TimeGPT* (Nixtla) is similar but only available as a paid API.

**Agenda**

1. Load the sample data from notebook 2_8 (daily revenue of 10 products)
2. Zero-shot forecasts with Chronos-Bolt and Chronos-2
3. Chronos-2 with cross-learning and with a covariate
4. Comparison with the NHITS models from notebook 2_8
5. Final forecast for the next 4 weeks

## Setup

In [ ]:
# Install the required packages (Chronos is not pre-installed on Colab)
%pip install -q pandas numpy matplotlib torch chronos-forecasting

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from chronos import BaseChronosPipeline, Chronos2Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # a GPU is not necessary, the CPU is fast enough here
print("device:", DEVICE)

## 1. Load the data

We use the files created in notebook 2_8:

* `online_retail_assortment_daily_sales.csv` – daily revenue (GBP) of the 10 most regularly sold products of the *Online Retail* data set,
  in long format with the columns `ds` (date), `ID` (product = stock code) and `y` (revenue),
* `nhits_test_predictions.csv` – the forecasts of the NHITS models for the test period, for the comparison at the end.

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sciform/sciform-hwz-ai-in-controlling/main/data/online_retail_assortment_daily_sales.csv
!wget -q -nc https://raw.githubusercontent.com/sciform/sciform-hwz-ai-in-controlling/main/data/nhits_test_predictions.csv

In [ ]:
sales = pd.read_csv("online_retail_assortment_daily_sales.csv", parse_dates=["ds"], dtype={"ID": str})
nhits_pred = pd.read_csv("nhits_test_predictions.csv", parse_dates=["ds"], dtype={"ID": str})

products = sales["ID"].unique()
print(sales.shape, "-", len(products), "products,", sales["ds"].min().date(), "–", sales["ds"].max().date())
sales.head()

### Same train / test split as in notebook 2_8

The last **28 days** (12 Nov – 9 Dec 2011) are the test period. For Chronos the training part is not used for training but as **context**:
the model "reads" the history and continues it.

In [ ]:
HORIZON = 28
QUANTILES = [0.1, 0.5, 0.9]          # 80% prediction interval + median

cutoff = sales["ds"].max() - pd.Timedelta(days=HORIZON)
df_context = sales[sales["ds"] <= cutoff].reset_index(drop=True)
df_test = sales[sales["ds"] > cutoff].reset_index(drop=True)
print(f"context: {df_context['ds'].min().date()} – {df_context['ds'].max().date()}  ({df_context['ds'].nunique()} days)")
print(f"test:    {df_test['ds'].min().date()} – {df_test['ds'].max().date()}  ({df_test['ds'].nunique()} days)")

In [ ]:
def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    err = y_true - y_pred
    return pd.Series({"MAE": np.mean(np.abs(err)),
                      "RMSE": np.sqrt(np.mean(err ** 2)),
                      "WAPE_%": 100 * np.sum(np.abs(err)) / np.sum(np.abs(y_true))})

def evaluate(pred, col="yhat", name="model"):
    # metrics per product + over all products
    per_id = pred.groupby("ID").apply(lambda g: metrics(g["y"], g[col]))
    per_id.loc["ALL"] = metrics(pred["y"], pred[col])
    per_id.columns = pd.MultiIndex.from_product([[name], per_id.columns])
    return per_id

results, predictions = {}, {}

def register(name, pred):
    # store a forecast (columns ds, ID, y, yhat [, lo, hi]) and print its overall accuracy
    predictions[name] = pred
    results[name] = evaluate(pred, name=name)
    return results[name].loc[["ALL"]]

### Baseline: seasonal naive forecast

As in notebook 2_8: the revenue of the last observed week is repeated for the whole horizon.

In [ ]:
last_week = df_context[df_context["ds"] > cutoff - pd.Timedelta(days=7)].assign(dow=lambda d: d["ds"].dt.dayofweek)
pred_naive = (df_test.assign(dow=lambda d: d["ds"].dt.dayofweek)
              .merge(last_week[["ID", "dow", "y"]].rename(columns={"y": "yhat"}), on=["ID", "dow"]))
register("seasonal_naive", pred_naive)

## 2. Zero-shot forecast with Chronos-Bolt

`from_pretrained` downloads the model weights from Hugging Face (Chronos-Bolt *small*: 48 M parameters, ~200 MB).
`predict_df` takes a data frame in long format – exactly the format of our sample data – and returns for every product and day the
point forecast (`predictions` = median) and the requested quantiles.

**No training, no hyperparameters, no feature engineering.**

In [ ]:
def chronos_forecast(pipeline, context, **kwargs):
    fc = pipeline.predict_df(context, id_column="ID", timestamp_column="ds", target="y",
                             prediction_length=HORIZON, quantile_levels=QUANTILES, **kwargs)
    fc = fc.rename(columns={"predictions": "yhat", "0.1": "lo", "0.9": "hi"})[["ds", "ID", "yhat", "lo", "hi"]]
    fc[["yhat", "lo", "hi"]] = fc[["yhat", "lo", "hi"]].clip(lower=0)      # revenue cannot be negative
    return fc

In [ ]:
%%time
bolt = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", device_map=DEVICE)

fc_bolt = chronos_forecast(bolt, df_context)
fc_bolt.head()

In [ ]:
register("chronos_bolt", df_test.merge(fc_bolt, on=["ds", "ID"]))

## 3. Chronos-2

### 3.1 Zero-shot, each product on its own

Chronos-2 is used in exactly the same way. Each product is forecast independently (`cross_learning=False`, the default).

In [ ]:
%%time
chronos2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

fc_c2 = chronos_forecast(chronos2, df_context)
register("chronos2", df_test.merge(fc_c2, on=["ds", "ID"]))

### 3.2 Cross-learning: a zero-shot *global* model

With `cross_learning=True` all products of the batch are forecast **jointly**: the model's attention can look at the other
products of the assortment – similar to the global model of notebook 2_8, but without any training.
According to the Chronos-2 authors this helps most when the single series are short.

In [ ]:
fc_c2_cl = chronos_forecast(chronos2, df_context, cross_learning=True)
register("chronos2_cross", df_test.merge(fc_c2_cl, on=["ds", "ID"]))

### 3.3 Known future covariate: "shop does not ship on Saturdays"

Chronos-2 can use additional columns as covariates. Columns that are also contained in `future_df` are treated as
**known in the future** (calendar effects, holidays, planned promotions, prices, ...).
As an example we add a flag `closed` for Saturdays, the day without shipping.

In [ ]:
def add_calendar(d):
    return d.assign(closed=(d["ds"].dt.dayofweek == 5).astype(float))

future_cov = add_calendar(df_test[["ID", "ds"]])            # only the covariate, of course not y!
fc_c2_cov = chronos_forecast(chronos2, add_calendar(df_context), future_df=future_cov, cross_learning=True)
register("chronos2_cross_cov", df_test.merge(fc_c2_cov, on=["ds", "ID"]))

## 4. Comparison with NHITS (notebook 2_8)

We add the forecasts of the three NHITS models that were trained in notebook 2_8 on exactly the same data.

In [ ]:
for col in ["nhits_local", "nhits_global", "nhits_global_cal"]:
    pred = df_test.merge(nhits_pred[["ds", "ID", col]].rename(columns={col: "yhat"}), on=["ds", "ID"])
    register(col, pred)

comparison = pd.concat(results.values(), axis=1)
overall = comparison.loc["ALL"].unstack(0).T.sort_values("WAPE_%")
overall["type"] = ["GenAI zero-shot" if i.startswith("chronos") else "trained" if i.startswith("nhits") else "baseline"
                   for i in overall.index]
overall

In [ ]:
colors = {"GenAI zero-shot": "tab:purple", "trained": "tab:blue", "baseline": "tab:grey"}
ax = overall["WAPE_%"].plot.barh(color=[colors[t] for t in overall["type"]], figsize=(8, 4),
                                 title="WAPE over the whole assortment, 28-day horizon (lower is better)")
ax.invert_yaxis(); ax.set_xlabel("WAPE in %")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], labels=list(colors), loc="lower right")
plt.show()

### Accuracy per product (WAPE in %)

In [ ]:
wape = comparison.xs("WAPE_%", axis=1, level=1)[overall.index]
wape.style.format(precision=1).highlight_min(axis=1, color="lightgreen")

### Forecast vs. actuals per product

In [ ]:
best_genai = overall[overall["type"] == "GenAI zero-shot"].index[0]
hist = df_context[df_context["ds"] > cutoff - pd.Timedelta(days=42)]

fig, axes = plt.subplots(5, 2, figsize=(16, 13), sharex=True)
for ax, i in zip(axes.flat, products):
    h = hist[hist["ID"] == i]
    g = predictions[best_genai].query("ID == @i")
    n = predictions["nhits_global"].query("ID == @i")
    ax.plot(h["ds"], h["y"], color="grey", lw=1, label="history")
    ax.plot(g["ds"], g["y"], color="black", lw=1.2, label="actual")
    ax.plot(g["ds"], g["yhat"], color="tab:purple", label=best_genai)
    ax.fill_between(g["ds"], g["lo"], g["hi"], color="tab:purple", alpha=0.2, label="80% interval")
    ax.plot(n["ds"], n["yhat"], color="tab:blue", ls="--", label="nhits_global (2_8)")
    ax.axvline(cutoff, color="red", lw=0.8)
    ax.set_title(f"product {i}", fontsize=10); ax.grid(alpha=0.3)
axes.flat[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Daily revenue: GenAI zero-shot vs. trained NHITS (GBP)", fontsize=14)
fig.tight_layout(); plt.show()

### Controlling view: total revenue of the assortment

Product forecasts are summed up (bottom-up) and compared with the actual revenue of the test period.

In [ ]:
total = pd.DataFrame({name: p.groupby("ds")["yhat"].sum() for name, p in predictions.items()})
total.insert(0, "actual", df_test.groupby("ds")["y"].sum())

ax = total[["actual", best_genai, "nhits_global", "seasonal_naive"]].plot(
    figsize=(14, 4), style=["-", "-", "--", ":"], color=["black", "tab:purple", "tab:blue", "tab:green"],
    title="Daily revenue of the assortment (GBP)")
ax.grid(alpha=0.3); plt.show()

summary = pd.DataFrame({
    "forecast_GBP": total.drop(columns="actual").sum(),
    "deviation_%": 100 * (total.drop(columns="actual").sum() / total["actual"].sum() - 1),
    "daily_WAPE_%": [metrics(total["actual"], total[c])["WAPE_%"] for c in total.columns[1:]],
}).loc[overall.index]
print(f"actual revenue of the test period: {total['actual'].sum():,.0f} GBP")
summary

### Interpretation

* **Without any training** the foundation model reaches the accuracy of the trained NHITS models of notebook 2_8.
  This is remarkable: it has never seen this shop, it only "reads" one year of history per product.
* Chronos-2 is more accurate than the older and smaller Chronos-Bolt.
* **Cross-learning** and the **Saturday covariate** do *not* help here: every product already has a full year of history, and the
  Saturday zeros are clearly visible in the context anyway. Covariates are more useful for information that cannot be seen in the
  history (planned promotions, price changes, new holidays). → Always test such options against a baseline instead of assuming they help.
* Like NHITS, the foundation models **underestimate the total revenue** of the pre-Christmas peak by about a third: the
  context contains only one Christmas season and the increase from November on is not visible in the history before the cut-off.
  The seasonal naive forecast gets the total right (but is very inaccurate per day and product).
* Advantages of foundation models: no training pipeline, quick to set up, works for new products with short history, prediction intervals
  out of the box. Disadvantages: large models (GPU for big assortments), hardly interpretable (like NHITS), domain
  knowledge can only be added via covariates or fine-tuning.

## 5. Final forecast for the next 4 weeks

For the forecast into the future we simply give the model the **complete** history as context.

In [ ]:
best_c2 = chronos2 if best_genai.startswith("chronos2") else bolt
forecast = chronos_forecast(best_c2, sales)
forecast.columns = ["ds", "ID", "forecast", "lower_80", "upper_80"]
forecast.to_csv("assortment_sales_forecast_chronos.csv", index=False)
forecast.head(10)

In [ ]:
# planned revenue per product and week
plan = forecast.assign(week=forecast["ds"].dt.to_period("W")).pivot_table(
    index="ID", columns="week", values="forecast", aggfunc="sum")
plan["total"] = plan.sum(axis=1)
plan.loc["ALL"] = plan.sum()
plan.round(0)

## Summary and exercises

* Time series foundation models apply the GenAI idea to forecasting: one large pretrained model, used **zero-shot** for any time series.
* With `chronos-forecasting` a forecast for a whole assortment takes three lines of code and runs on a CPU.
* On our data the zero-shot forecast is as accurate as a trained global NHITS model – but neither of them knows about Christmas.
  Domain knowledge and a critical look at the total (bias) remain the job of the controller.

**Exercises**

1. Try other model sizes: `amazon/chronos-bolt-tiny`, `amazon/chronos-bolt-mini`, `amazon/chronos-bolt-base`. How do accuracy and run time change?
2. Shorten the context to the last 60 days (`df_context.groupby("ID").tail(60)`). Does cross-learning help now?
3. Build a covariate that is *not* visible in the history, e.g. a flag for the weeks before Christmas, and test it.
4. Combine the forecasts of `chronos2` and `nhits_global` (mean of both, an *ensemble*). Is the ensemble better than each model alone?
5. Fine-tune Chronos-2 on the context data: `chronos2.fit(inputs=..., prediction_length=HORIZON, finetune_mode="lora")`
   (see the [Chronos documentation](https://github.com/amazon-science/chronos-forecasting)).
6. Replace Chronos with another open model, e.g. [TimesFM](https://github.com/google-research/timesfm) or IBM's
   [Granite TTM](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2).